In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [2]:
lstm = nn.LSTM(3,3) # input dim 3 and output dim 3
inputs = [torch.rand(1, 3) for _ in range(5)]

# sequence, mini batch, elements
hidden = (torch.rand(1, 1, 3), torch.rand(1, 1, 3))

print("Inputs before reshape ", inputs)
for i in inputs:
    out, hidden = lstm(i.view(1, 1, -1), hidden)

inputs = torch.cat(inputs).view(len(inputs), 1, -1)
print("Inputs after reshape", inputs)
hidden = (torch.rand(1, 1, 3), torch.rand(1, 1, 3))
out, hidden = lstm(inputs, hidden)
print("Out")
print(out)
print("Hidden")
print(hidden)

Inputs before reshape  [tensor([[0.6705, 0.5896, 0.2873]]), tensor([[0.3486, 0.9579, 0.4075]]), tensor([[0.7819, 0.7165, 0.1768]]), tensor([[0.0748, 0.9799, 0.5261]]), tensor([[0.8427, 0.6036, 0.6608]])]
Inputs after reshape tensor([[[0.6705, 0.5896, 0.2873]],

        [[0.3486, 0.9579, 0.4075]],

        [[0.7819, 0.7165, 0.1768]],

        [[0.0748, 0.9799, 0.5261]],

        [[0.8427, 0.6036, 0.6608]]])
Out
tensor([[[ 0.2419,  0.0165,  0.0664]],

        [[ 0.1421, -0.0886,  0.0233]],

        [[ 0.1534, -0.1527, -0.0253]],

        [[ 0.0460, -0.1658, -0.0431]],

        [[ 0.0372, -0.2387, -0.0491]]], grad_fn=<StackBackward0>)
Hidden
(tensor([[[ 0.0372, -0.2387, -0.0491]]], grad_fn=<StackBackward0>), tensor([[[ 0.0768, -0.4277, -0.2526]]], grad_fn=<StackBackward0>))


In [15]:
# An LSTM for Part-of-Speech Tagging
def prepare_sequence(seq, word_to_idx):
    idx = [word_to_idx[word] for word in seq]
    return torch.tensor(idx, dtype=torch.long)

training_data = [
    # Tags are: DET - determiner; NN - noun; V - verb
    # For example, the word "The" is a determiner
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]

word_to_ix = {}

for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

print("Word_to_ix")
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}  # Assign each tag with a unique index

EMBEDDING_DIM = 6
HIDDEN_DIM = 6

Word_to_ix
{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [16]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tags_size):
        super(LSTMTagger, self).__init__()

        self.hidden_dim = hidden_dim
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        self.hiddenTotag = nn.Linear(hidden_dim, tags_size)

    def forward(self, sentences):
        embends = self.word_embedding(sentences)
        lstm_out, _ = self.lstm(embends.view(len(sentences), 1, -1))
        tag_space = self.hiddenTotag(lstm_out.view(len(sentences), -1))
        tag_score = F.log_softmax(tag_space, dim=1)
        return tag_score


In [18]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tags))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Scores before training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_idx=word_to_ix)
    tag_scores = model(inputs)
    print("tag_scores")
    print(tag_scores) 

for epoch in range(300):
    for sentence, tags in training_data:

        model.zero_grad()

        sentence_in = prepare_sequence(sentence, word_to_idx=word_to_ix)
        targets = prepare_sequence(tags, word_to_idx=tag_to_ix)

        tag_scores = model(sentence_in)

        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

# Scores after training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print("tag_scores after training")
    print(tag_scores)

tag_scores
tensor([[-1.2072, -1.8741, -1.4818, -1.8861, -1.7802],
        [-1.2413, -1.9290, -1.4502, -1.8211, -1.7759],
        [-1.2591, -1.7587, -1.4907, -1.8289, -1.8450],
        [-1.2497, -1.7178, -1.4886, -1.8387, -1.9022],
        [-1.2289, -1.7640, -1.5205, -1.8450, -1.8364]])
tag_scores after training
tensor([[-1.1027, -1.0969, -1.6402, -2.6836, -2.6330],
        [-1.1927, -1.0067, -1.6551, -2.6448, -2.6729],
        [-1.2415, -0.9403, -1.6130, -2.7692, -2.8377],
        [-1.2181, -0.9647, -1.5846, -2.7970, -2.8635],
        [-1.2684, -0.8484, -1.7059, -2.8556, -2.9663]])
